# Yahoo and Polygon Options Example

This notebook shows two common option-data workflows for a single underlying:

- Yahoo Finance option chain discovery with `yfinance`
- Polygon contract discovery and single-contract snapshot lookup

Default underlying: `AAPL`

Notes:
- Yahoo can be rate-limited or return partial option data.
- Polygon requires `POLYGON_API_KEY` in your environment.


## Setup

In PowerShell before starting Jupyter:

```powershell
$env:POLYGON_API_KEY = "your-key-here"
```

If `yfinance` is missing in your notebook kernel:

```powershell
pip install yfinance pandas requests
```


In [ ]:
!pip install yfinance pandas requests

In [1]:
import os
from datetime import datetime

import pandas as pd
import requests
import yfinance as yf

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

UNDERLYING = "SPY"
# POLYGON_API_KEY = os.getenv("POLYGON_API_KEY")

# Yahoo Finance symbols for option chains (index options use different tickers than price feeds)
_YF_TICKER_MAP = {
    "SPX": "^SPX",    # CBOE SPX options  (^GSPC is price-only, no chain)
    "NDX": "^NDX",
    "RUT": "^RUT",
    "VIX": "^VIX",
    "DJX": "^DJI",
}
_yf_symbol = _YF_TICKER_MAP.get(UNDERLYING, UNDERLYING)

# Separate price-only map — ^GSPC has reliable price but no options chain
_YF_PRICE_MAP = {
    "SPX": "^GSPC",
    "NDX": "^NDX",
    "RUT": "^RUT",
    "DJX": "^DJI",
}
_yf_price_symbol = _YF_PRICE_MAP.get(UNDERLYING, UNDERLYING)

print({
    "underlying": UNDERLYING,
    "yf_options_symbol": _yf_symbol,
    "yf_price_symbol": _yf_price_symbol,
    # "polygon_key_present": bool(POLYGON_API_KEY),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
})


{'underlying': 'SPY', 'yf_options_symbol': 'SPY', 'yf_price_symbol': 'SPY', 'timestamp': '2026-08-26T15:43:39'}


## Yahoo: discover expirations and option chain


In [2]:
ticker = yf.Ticker(_yf_symbol)

try:
    expirations = list(ticker.options)
except Exception as exc:
    raise RuntimeError(
        f"Yahoo option discovery failed for {_yf_symbol}. This usually means rate limiting or an upstream Yahoo response issue."
    ) from exc

if not expirations:
    raise RuntimeError(
        f"Yahoo returned no option expirations for {_yf_symbol}. This often means rate limiting or missing upstream data."
    )

expiration_df = pd.DataFrame({"expiration": expirations})
expiration_df.head(10)


,expiration
0,2026-08-26
1,2026-08-27
2,2026-08-28
3,2026-08-31
4,2026-09-01
5,2026-09-02
6,2026-09-03
7,2026-09-04
8,2026-09-11
9,2026-09-18


In [ ]:
pd.head(10)

In [ ]:
selected_expiration = expirations[9]

try:
    chain = ticker.option_chain(selected_expiration)
except Exception as exc:
    raise RuntimeError(
        f"Yahoo option chain lookup failed for expiration {selected_expiration}."
    ) from exc

calls = chain.calls.copy()
puts = chain.puts.copy()

if calls.empty and puts.empty:
    raise RuntimeError("Yahoo returned an empty option chain for the selected expiration.")

print("selected_expiration:", selected_expiration)
print("calls:", len(calls), "puts:", len(puts))

calls.head(10)


In [ ]:
import math
from datetime import date as _date

_DELTA_TARGET  = 0.20
_DELTA_TOL     = 0.05   # keep |delta| in [0.15, 0.25]
_IV_FALLBACK   = 0.20
_RISK_FREE     = 0.05


def _ncdf(x):
    return 0.5 * math.erfc(-x / math.sqrt(2))


def _bs_delta(S, K, T, sigma, option_type, rf=_RISK_FREE):
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.0
    d1 = (math.log(S / K) + (rf + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    return _ncdf(d1) if option_type == "call" else _ncdf(d1) - 1.0


# underlying price
_pu = yf.Ticker(_yf_price_symbol)
_pl = getattr(_pu.fast_info, "last_price", None)
if not _pl:
    _ph = _pu.history(period="5d")["Close"].dropna()
    if _ph.empty:
        raise RuntimeError(f"Could not fetch price for {_yf_price_symbol}")
    _pl = float(_ph.iloc[-1])
_S = float(_pl)

_dte_yf = max((_date.fromisoformat(selected_expiration) - _date.today()).days, 1)
_T = _dte_yf / 365.0
print(f"underlying: {_S:.2f}   expiration: {selected_expiration}   dte: {_dte_yf}")

# compute delta for each row
def _add_delta(df, option_type):
    df = df.copy()
    df["delta"] = df.apply(
        lambda row: _bs_delta(_S, row["strike"], _T, row["impliedVolatility"] or _IV_FALLBACK, option_type),
        axis=1,
    ).round(4)
    return df

_calls_all = _add_delta(calls, "call")
_puts_all  = _add_delta(puts,  "put")

# filter: call delta ≈ +0.20, put |delta| ≈ 0.20
calls_d20 = _calls_all[(_calls_all["delta"] - _DELTA_TARGET).abs() <= _DELTA_TOL].reset_index(drop=True)
puts_d20  = _puts_all[(_puts_all["delta"].abs() - _DELTA_TARGET).abs() <= _DELTA_TOL].reset_index(drop=True)

print(f"calls with delta ≈ {_DELTA_TARGET} (±{_DELTA_TOL}): {len(calls_d20)}")
display(calls_d20[["contractSymbol", "strike", "delta", "bid", "ask", "impliedVolatility", "openInterest"]])

print(f"\nputs  with |delta| ≈ {_DELTA_TARGET} (±{_DELTA_TOL}): {len(puts_d20)}")
display(puts_d20[["contractSymbol", "strike", "delta", "bid", "ask", "impliedVolatility", "openInterest"]])


In [ ]:
puts.head(10)


## Yahoo: pick one contract and inspect it

Yahoo's chain already includes `contractSymbol`, which is usually the easiest contract identifier to reuse.


In [ ]:
sample_call = calls.sort_values("strike").iloc[0]
yahoo_contract_symbol = sample_call["contractSymbol"]

sample_call[[
    "contractSymbol",
    "strike",
    "lastPrice",
    "bid",
    "ask",
    "impliedVolatility",
    "openInterest",
    "volume",
]].to_frame(name="value")


In [ ]:
option_ticker = yf.Ticker(yahoo_contract_symbol)
option_history = option_ticker.history(period="5d", interval="1d")

print("yahoo_contract_symbol:", yahoo_contract_symbol)
print("history_rows:", len(option_history))
option_history.tail()


## Polygon helpers


In [ ]:
if not POLYGON_API_KEY:
    raise RuntimeError("Set POLYGON_API_KEY in your environment before running Polygon examples.")

POLYGON_BASE_URL = "https://api.polygon.io"

def polygon_get(path: str, **params):
    response = requests.get(
        f"{POLYGON_BASE_URL}{path}",
        params={**params, "apiKey": POLYGON_API_KEY},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


## Polygon: discover contracts for the same underlying and expiration

This uses Polygon's `GET /v3/reference/options/contracts` endpoint.


In [ ]:
from datetime import date

# Fetch distinct expirations from Polygon — limit=1000 and gte today to avoid buried results
_exp_json = polygon_get(
    "/v3/reference/options/contracts",
    underlying_ticker=UNDERLYING,
    contract_type="call",
    **{"expiration_date.gte": date.today().isoformat()},
    order="asc",
    sort="expiration_date",
    limit=1000,
)
_exp_results = _exp_json.get("results", [])
if not _exp_results:
    raise RuntimeError("Polygon returned no contracts for this underlying. Check UNDERLYING and POLYGON_API_KEY.")

polygon_expirations = sorted({r["expiration_date"] for r in _exp_results})
selected_expiration = polygon_expirations[min(9, len(polygon_expirations) - 1)]

print(f"found {len(polygon_expirations)} expirations:")
for exp in polygon_expirations:
    marker = " <-- selected" if exp == selected_expiration else ""
    print(f"  {exp}{marker}")


import requests

url = "https://api.massive.com/stocks/filings/vX/index"
params = {
    "ticker": "AAPL",
    "form_type": "10-K",
    "limit": 5,
    "sort": "filing_date.desc"
}

response = requests.get(url, params=params, headers={"Authorization": f"Bearer {API_KEY}"})
filings = response.json()["results"]

for filing in filings:
    print(f"{filing['filing_date']}  {filing['form_type']}  {filing['issuer_name']}")
    print(f"  {filing['filing_url']}")

In [ ]:
import requests
API_KEY = "eRpXsk9YhoXF1OiKEk2xlpsFrhZoAkh0"
url = "https://api.massive.com/stocks/filings/vX/index"
params = {
    "ticker": "AAPL",
    "form_type": "10-K",
    "limit": 5,
    "sort": "filing_date.desc"
}

response = requests.get(url, params=params, headers={"Authorization": f"Bearer {API_KEY}"})
filings = response.json()["results"]

for filing in filings:
    print(f"{filing['filing_date']}  {filing['form_type']}  {filing['issuer_name']}")
    print(f"  {filing['filing_url']}")

In [ ]:
import yfinance as yf

ticker = yf.Ticker("goog")

# Get current price info
info = ticker.info
print("Company:", info.get("longName"))
print("Current price:", info.get("currentPrice"))

# Get historical data
hist = ticker.history(period="5d")
print(hist)
# ticker.option_chain
print("Options expirations:", ticker.option_chain())

In [ ]:
import math
from datetime import date

DELTA_MAX = 0.21       # keep calls with delta < this
IV_FALLBACK = 0.20     # used when a contract has no IV in the chain


def _ncdf(x: float) -> float:
    """Standard normal CDF via math.erfc — no scipy needed."""
    return 0.5 * math.erfc(-x / math.sqrt(2))


def _bs_delta_call(S: float, K: float, T: float, sigma: float, r: float = 0.05) -> float:
    """Black-Scholes delta for a European call."""
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.0
    d1 = (math.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    return _ncdf(d1)


# --- underlying price via yfinance (use price symbol, not options symbol) ---
_uf = yf.Ticker(_yf_price_symbol)
_last = getattr(_uf.fast_info, "last_price", None)
if not _last:
    _hist = _uf.history(period="5d")["Close"].dropna()
    if _hist.empty:
        raise RuntimeError(f"Could not fetch underlying price for {_yf_price_symbol}.")
    _last = float(_hist.iloc[-1])
underlying_price = float(_last)
print(f"underlying_price ({_yf_price_symbol}): {underlying_price}")

# --- fetch all contracts for the selected expiration ---
_page_results: list = []
_next_url: str | None = None
while True:
    if _next_url:
        _r = requests.get(_next_url, params={"apiKey": POLYGON_API_KEY}, timeout=30)
        _r.raise_for_status()
        _page = _r.json()
    else:
        _page = polygon_get(
            "/v3/reference/options/contracts",
            underlying_ticker=UNDERLYING,
            expiration_date=selected_expiration,
            contract_type="call",
            order="asc",
            sort="strike_price",
            limit=1000,
        )
    _page_results.extend(_page.get("results", []))
    _next_url = _page.get("next_url")
    if not _next_url:
        break

if not _page_results:
    raise RuntimeError("No contracts returned for the selected expiration.")

# --- compute delta for each contract ---
_dte = (date.fromisoformat(selected_expiration) - date.today()).days
T = max(_dte, 1) / 365.0

_rows = []
for c in _page_results:
    K = float(c["strike_price"])
    iv = float(c.get("implied_volatility") or IV_FALLBACK)
    delta = _bs_delta_call(underlying_price, K, T, iv)
    _rows.append({
        "ticker":          c.get("ticker"),
        "expiration_date": c.get("expiration_date"),
        "contract_type":   c.get("contract_type"),
        "exercise_style":  c.get("exercise_style"),
        "strike_price":    K,
        "iv_used":         round(iv, 4),
        "delta":           round(delta, 4),
    })

_all_contracts = pd.DataFrame(_rows)

# Filter: delta < DELTA_MAX
polygon_contracts = (
    _all_contracts[_all_contracts["delta"] < DELTA_MAX]
    .reset_index(drop=True)
)

print(f"{len(_all_contracts)} total calls → {len(polygon_contracts)} with delta < {DELTA_MAX}")
polygon_contracts[["ticker", "strike_price", "delta", "iv_used", "exercise_style"]]


In [ ]:
# Enrich with bid/ask from Yahoo Finance (Polygon reference endpoint has no quotes)
_yf_ticker = yf.Ticker(_yf_symbol)
try:
    _yf_chain = _yf_ticker.option_chain(selected_expiration)
    _yf_calls = _yf_chain.calls[["strike", "bid", "ask", "lastPrice", "impliedVolatility", "openInterest"]].copy()
    _yf_calls = _yf_calls.rename(columns={
        "strike":           "strike_price",
        "lastPrice":        "last_price",
        "impliedVolatility":"iv_yahoo",
        "openInterest":     "open_interest",
    })
    polygon_contracts = polygon_contracts.merge(_yf_calls, on="strike_price", how="left")
    print(f"merged Yahoo bid/ask for {selected_expiration}: {_yf_calls['strike_price'].nunique()} strikes available")
except Exception as exc:
    print(f"Yahoo chain fetch failed ({exc}); bid/ask columns will be missing")

cols = ["ticker", "strike_price", "delta", "iv_used", "bid", "ask", "last_price", "open_interest"]
polygon_contracts[[c for c in cols if c in polygon_contracts.columns]]


## Polygon: fetch a snapshot for one option contract

This uses Polygon's `GET /v3/snapshot/options/{underlyingAsset}/{optionContract}` endpoint.


In [ ]:
polygon_contract_symbol = polygon_contracts.iloc[0]["ticker"]
polygon_snapshot = polygon_get(f"/v3/snapshot/options/{UNDERLYING}/{polygon_contract_symbol}")
polygon_snapshot


In [ ]:
snapshot_results = polygon_snapshot.get("results", {})
snapshot_summary = {
    "ticker": snapshot_results.get("details", {}).get("ticker"),
    "expiration_date": snapshot_results.get("details", {}).get("expiration_date"),
    "strike_price": snapshot_results.get("details", {}).get("strike_price"),
    "contract_type": snapshot_results.get("details", {}).get("contract_type"),
    "break_even_price": snapshot_results.get("break_even_price"),
    "implied_volatility": snapshot_results.get("implied_volatility"),
    "open_interest": snapshot_results.get("open_interest"),
    "delta": snapshot_results.get("greeks", {}).get("delta"),
    "gamma": snapshot_results.get("greeks", {}).get("gamma"),
    "theta": snapshot_results.get("greeks", {}).get("theta"),
    "vega": snapshot_results.get("greeks", {}).get("vega"),
    "last_quote_bid": snapshot_results.get("last_quote", {}).get("bid"),
    "last_quote_ask": snapshot_results.get("last_quote", {}).get("ask"),
    "underlying_price": snapshot_results.get("underlying_asset", {}).get("price"),
}

pd.DataFrame([snapshot_summary])


## Quick comparison


In [ ]:
comparison_df = pd.DataFrame([
    {
        "provider": "Yahoo",
        "contract": yahoo_contract_symbol,
        "expiration": selected_expiration,
        "strike": sample_call.get("strike"),
        "last_price": sample_call.get("lastPrice"),
        "bid": sample_call.get("bid"),
        "ask": sample_call.get("ask"),
        "implied_volatility": sample_call.get("impliedVolatility"),
        "open_interest": sample_call.get("openInterest"),
    },
    {
        "provider": "Polygon",
        "contract": snapshot_summary.get("ticker"),
        "expiration": snapshot_summary.get("expiration_date"),
        "strike": snapshot_summary.get("strike_price"),
        "last_price": None,
        "bid": snapshot_summary.get("last_quote_bid"),
        "ask": snapshot_summary.get("last_quote_ask"),
        "implied_volatility": snapshot_summary.get("implied_volatility"),
        "open_interest": snapshot_summary.get("open_interest"),
    },
])

comparison_df


## Source references

- yfinance docs: `Ticker.options` and `Ticker.option_chain(...)`
- Polygon contracts endpoint: `GET /v3/reference/options/contracts`
- Polygon contract snapshot endpoint: `GET /v3/snapshot/options/{underlyingAsset}/{optionContract}`
